# 비만클리닉 doctor agent 만들기
**대한의료정보학회 2026 튜토리얼 실습** · 모든 환자 데이터는 합성 데이터입니다.

오늘은 **2026년 11월 25일**, 비만클리닉 진료일입니다. 세 환자가 옵니다.

| 환자 | 오늘 할 일 | 에이전트가 쓸 것 |
|---|---|---|
| **P1** (남, 27세) | 삭센다 4개월째 재진 | **재진 전 요약**: 진료실 들어가기 전 30초 안에 읽을 요약 |
| **P2** (여, 40세) | 처음 온 환자 | **초진기록 초안**: 의사 메모 + 문진표 + 기존 기록으로 |
| **P3** (남, 35세) | 위고비를 중단한 환자의 마지막 진료 | **치료 종료 기록**: 치료 전체를 지침 기준으로 정리 |

세 장면 모두 **같은 에이전트, 같은 도구**를 쓰고 **양식만 바꿉니다**.

코드는 모두 채워져 있습니다. 위에서부터 한 셀씩 실행하면서 결과와 주석(`#`)을 읽어 가면 됩니다.
🔁 표시가 있는 셀은 고쳐서 다시 돌려 보면 좋은 곳입니다(선택).

| 순서 | 내용 |
|---|---|
| 0 | 준비: 자료 내려받기, API 키 |
| 1 | LLM 한 번 불러 보기 |
| 2 | 데이터 보기: 왜 그대로 넣지 않는가 |
| 3 | 도구: 읽기와 계산은 코드가 한다 |
| 4 | 에이전트 루프: 반복문 하나 (P1 재진 요약) |
| 5 | 결과 점검하기, 프롬프트 한 줄의 힘 |
| 6 | 초진기록: 양식만 바꾸기 (P2) |
| 7 | 치료 종료 기록 (P3) |
| 8 | 채팅 화면으로 써 보기 |

## 0. 준비

**① 이 노트북을 내 Drive에 사본으로 저장하세요** (메뉴 `파일 → Drive에 사본 저장`). 그래야 고친 내용이 남습니다.

**② API 키 등록**: 왼쪽 막대의 🔑(보안 비밀) → `새 보안 비밀 추가` → 이름 `GEMINI_API_KEY`, 값에 키 붙여넣기 → `노트북 액세스` 켜기.
키는 [Google AI Studio](https://aistudio.google.com/apikey)에서 무료로 만듭니다(결제 정보 필요 없음). **키를 코드 셀에 직접 붙여넣지 마세요.** 노트북을 공유할 때 같이 나갑니다.

첫 셀을 실행하면 "Google에서 작성하지 않은 노트북" 경고가 뜹니다. `무시하고 계속`을 누르세요.

In [ ]:
# 실습 자료(코드·합성 데이터)를 GitHub에서 내려받아 /content/kosmi 에 풉니다. 30초 정도 걸립니다.
import os, sys, zipfile, urllib.request

REPO_ZIP = "https://github.com/chopresent/kosmi-tutorial-doctor-agent/archive/refs/heads/main.zip"
PKG = os.environ.get("KOSMI_PKG_DIR", "/content/kosmi")           # 자료를 풀 폴더

if not os.path.exists(os.path.join(PKG, "kosmi_tools.py")):         # 이미 받았으면 건너뜀
    urllib.request.urlretrieve(REPO_ZIP, "/content/kosmi.zip")
    with zipfile.ZipFile("/content/kosmi.zip") as z:
        z.extractall("/content")
        top = z.namelist()[0].split("/")[0]                          # 압축 안의 최상위 폴더 이름
    os.rename(os.path.join("/content", top), PKG)
sys.path.insert(0, PKG)                                              # 실습 모듈을 import 할 수 있게

!pip -q install openai gradio > /dev/null
print("준비 완료:", sorted(f for f in os.listdir(PKG) if not f.startswith(".")))

In [ ]:
# 🔑 보안 비밀에 넣어 둔 API 키를 환경 변수로 불러옵니다.
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("🔑 키를 불러왔습니다.")
except ImportError:                     # Colab이 아닌 곳에서 실행할 때
    if not os.environ.get("GEMINI_API_KEY"):
        import getpass
        os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
except Exception as e:
    print("⚠️ 키를 찾지 못했습니다. 왼쪽 🔑에 GEMINI_API_KEY를 넣고 '노트북 액세스'를 켰는지 확인하세요.", e)

## 1. LLM 한 번 불러 보기

Gemini를 **OpenAI 호환 방식**으로 부릅니다. 원내 서버(vLLM 등)로 옮길 때는 아래 `base_url`, `api_key`, 모델 이름 **세 가지만** 바뀝니다.
무료 등급 한도는 모델별로 **분당 15회, 하루 500회**입니다. 이 노트북 전체를 돌려도 15회 안팎입니다.

In [ ]:
from openai import OpenAI
import kosmi_tools as kt      # 데이터, 도구
import kosmi_agent as ka      # 모델 호출, 프롬프트·양식, 에이전트 루프, 자동 점검, 채팅 화면

client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",   # 원내 vLLM이면 "http://내부서버:8000/v1"
    api_key=os.environ["GEMINI_API_KEY"],
)
print("사용 모델:", ka.MODELS)   # 첫 모델의 하루 한도가 끝나거나 응답이 45초를 넘기면 다음 모델로 자동 전환

In [ ]:
# 🔁 질문을 바꿔서 다시 실행해 보세요.
# ka.chat()은 client.chat.completions.create()를 감싼 함수입니다.
# 분당 한도(429)에 걸리면 잠시 쉬었다가 다시 부르고, 하루 한도가 끝나면 다른 무료 모델로 넘어갑니다.
r = ka.chat(client, [{"role": "user", "content": "GLP-1 유사체 비만약의 흔한 부작용을 세 가지만 한 줄씩 알려줘"}])
print(r.choices[0].message.content)

## 2. 데이터 보기: 왜 그대로 넣지 않는가

세 환자의 기록이 표 7개로 들어 있습니다.

| 표 | 내용 |
|---|---|
| `patients` | 성별, 출생연도 |
| `diagnoses` | 진단 (KCD 코드) |
| `prescriptions` | 비만약 처방: 약, 용량, 처방일수 |
| `vitals` | 체중·키·BMI·혈압 (측정 1회 = 1행) |
| `questionnaire` | 환자가 작성한 체중관리 문진표 21문항 |
| `notes` | 이전 진료기록(초진기록)의 자유서술 |
| `labs` | 혈액검사 결과 |

In [ ]:
db = kt.load_data()                              # 표 7개를 dict로 읽음: db["vitals"] 등
for name, df in db.items():
    print(f"{name:14s} {len(df):4d}행   열: {list(df.columns)}")

In [ ]:
# 오늘 첫 환자 P1의 기록을 표마다 모아 봅니다.
rows = {name: df[df["patient_id"] == "P1"] for name, df in db.items()}
print({name: len(r) for name, r in rows.items()})

rows["vitals"].tail(8)          # 활력징후 마지막 8행: 체중은 측정한 날만, 혈압은 하루에 여러 번 잰 날도 있음

이 정도면 통째로 LLM에 넣어도 될 것 같지만, 그렇게 하지 않습니다.
1. **필요한 것만 골라야 한다**: 실습 데이터는 작지만, 실제 EMR은 수년간의 진료·검사·간호기록이 쌓여 금방 수십만 자가 됩니다.
2. **계산을 LLM이 하게 된다**: 체중 변화율, 증량 단계, 지침 판정을 LLM이 표를 보고 암산하면 틀릴 수 있습니다.

그래서 **읽기와 계산은 코드(도구)가 하고, LLM은 어떤 도구를 쓸지 고르고 결과를 글로 정리만** 하게 나눕니다.

## 3. 도구: 읽기와 계산은 코드가 한다

`kt.Context("P1")`은 진료할 환자 한 명의 기록을 엽니다. 도구는 이 환자에게 **묶여** 있어서, LLM이 다른 환자의 기록을 볼 방법이 없습니다.

In [ ]:
ctx = kt.Context("P1")            # 오늘 첫 환자 P1을 연다 (P1의 오늘 할 일 = 재진 요약)
print(ctx.header())
print()
print(kt.get_weight_trend(ctx))   # 체중 도구: 기준 체중, 변화율, 최저 체중을 코드가 계산

도구 결과에는 이미 **출처 태그**(`[기록 2026-11-14]`)가 붙어 있습니다. LLM은 이 태그를 옮겨 적기만 하면 됩니다.

아래는 대한비만학회 2022 지침을 바탕으로 한 **판정 도구**입니다. "유지 용량 3개월 시점에 5% 이상 감량했는가"를 정의대로 계산합니다.
🔁 `get_medication_timeline`, `get_questionnaire`, `get_previous_notes`, `get_labs`로 바꿔서도 실행해 보세요.

In [ ]:
print(kt.assess_response(ctx))

### LLM에게 도구를 알려 주기
LLM은 도구를 **실행하지 못합니다**. 도구의 *이름과 설명*만 보고 "이걸 불러 달라"고 요청할 뿐입니다. 그래서 **설명(description)이 곧 사용 설명서**입니다.

In [ ]:
# 도구 명세 하나: 이름, 설명, 인자 형식(JSON Schema). 환자는 이미 열려 있으니 인자가 없습니다.
weight_tool = {
    "type": "function",
    "function": {
        "name": "get_weight_trend",
        "description": "체중 추이: 기준 체중, 최근 체중, 변화율(%), 최저 체중, BMI. 체중·변화율은 반드시 이 도구 값만 쓴다.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
}

# 질문과 함께 도구 목록(tools=)을 넘기면, LLM은 답 대신 '도구 호출 요청'을 돌려줄 수 있습니다.
r = ka.chat(client,
            [{"role": "user", "content": "이 환자의 체중이 등록 후 얼마나 줄었나요?"}],
            tools=[weight_tool])
msg = r.choices[0].message
print("LLM의 답:", msg.content)                                                       # 보통 비어 있음
print("LLM이 요청한 도구:", [(tc.function.name, tc.function.arguments) for tc in (msg.tool_calls or [])])

LLM은 답 대신 **도구 호출 요청**만 돌려줬습니다. 실제 실행과 결과 전달은 우리 코드의 몫입니다. 이 왕복을 자동으로 반복하는 것이 **에이전트 루프**입니다.

실습에는 도구 8개를 만들어 두었습니다.

In [ ]:
for t in kt.TOOL_SPECS:
    print(f"- {t['function']['name']}: {t['function']['description'][:60]}")

## 4. 에이전트 루프: 반복문 하나

```
  ┌──────────────► LLM 호출 ──────────────┐
  │                                       ▼
  │                         도구 호출 요청이 있나?
  │                          │ 예            │ 아니오
  │           도구 실행 → 결과를 대화에 추가    └──► 최종 답 반환
  └───────────────────────────┘
```
아래가 에이전트 전부입니다. 줄마다 주석을 읽어 보세요.

In [ ]:
def my_agent(ctx, messages, tools=kt.TOOL_SPECS, max_steps=8):
    trace = []                                                  # 어떤 도구를 불렀는지 기록 (나중에 점검용)
    for step in range(1, max_steps + 1):                        # 무한 반복을 막기 위해 최대 8번
        r = ka.chat(client, messages, tools=tools)              # ① 대화 전체 + 도구 목록을 LLM에 보냄
        msg = r.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))      # ② LLM의 응답을 대화에 그대로 덧붙임

        if not msg.tool_calls:                                  # ③ 도구 요청이 없으면 = 최종 답
            print(f"✅ {step}번째 호출에서 답변 완료")
            return msg.content, trace

        for tc in msg.tool_calls:                               # ④ 요청한 도구를 하나씩 실행 (여러 개를 한꺼번에 요청하기도 함)
            result = kt.call_tool(ctx, tc.function.name, tc.function.arguments)   # 도구는 ctx(지금 연 환자)에 묶여 있음
            trace.append((step, tc.function.name, tc.function.arguments, result))
            print(f"🔧 [{step}] {tc.function.name}  →  {result.splitlines()[1][:60]}")
            messages.append({"role": "tool", "tool_call_id": tc.id,      # ⑤ 결과를 '어느 요청에 대한 답인지' 표시해 돌려줌
                             "content": result})
    return "(단계 초과)", trace

### 첫 실행: P1 재진 전 요약
`ka.build_messages()`는 **시스템 프롬프트**(규칙·지침 요점)와 **재진 요약 양식**, 요청 문장을 묶어 줍니다. 🔧 줄이 에이전트가 스스로 도구를 고른 기록입니다.

In [ ]:
print(ka.SYSTEM_COMMON)      # 시스템 프롬프트: 출처 태그 규칙, 계산값은 도구 값만, 지침 요점

In [ ]:
ctx = kt.Context("P1")
messages = ka.build_messages(ctx)          # 시스템 프롬프트 + 재진 요약 양식 + 요청
out, trace = my_agent(ctx, messages)
print()
print(out)

**살펴볼 것**
- 도구를 몇 번, 어떤 순서로 불렀나요? 보통 첫 호출에서 도구를 한꺼번에(병렬로) 요청하고, 두 번째 호출에서 글을 씁니다.
- 숫자 하나를 골라 출처 태그의 날짜를 아래 도구 결과에서 찾아보세요.

In [ ]:
ka.show_trace(trace)            # 도구 결과 앞부분. 전체를 보려면 ka.show_trace(trace, full=True)

## 5. 결과 점검하기, 프롬프트 한 줄의 힘

기록 문서는 정답이 하나가 아닙니다. 대신 **사실 추적성**은 자동으로 점검할 수 있습니다.
- **출처 표시**: 숫자가 있는 줄마다 출처 태그가 있나
- **날짜 근거**: 오늘 이후의 날짜를 지어내지 않았나 (도구가 계산한 처방 소진일 등은 통과)
- **수치 근거**: 쓴 숫자가 도구 결과·입력에 실제로 있나

In [ ]:
ka.print_check(ka.check_output(ctx, out, trace))

P1은 BMI 41의 3단계 비만인데 약물치료에 반응이 없었습니다. 시스템 프롬프트의 **[지침 요점]**에는 비만대사수술 상담 기준이 한 줄 들어 있습니다.
그 줄을 빼고 돌리면 "오늘 확인할 것"이 어떻게 달라지는지 보세요.

In [ ]:
# 시스템 프롬프트에서 수술 상담 기준 한 줄만 지운 버전
surgery_line = "- BMI 35 이상, 또는 30 이상이면서 동반질환이 있는데 생활습관·약물치료로 감량이 부족하면 비만대사수술 상담을 고려한다.\n"
no_surgery = ka.SYSTEM_COMMON.replace(surgery_line, "")

out2, trace2 = my_agent(ctx, ka.build_messages(ctx, system=no_surgery))
print("── 지침 줄 있음 ──"); print(out[out.find("오늘 확인할 것"):])
print("── 지침 줄 없음 ──"); print(out2[out2.find("오늘 확인할 것"):])

모델이 의학 지식이 없어서가 아니라, **우리 기관이 무엇을 확인하길 원하는지**를 프롬프트로 알려 주지 않았기 때문입니다.

## 6. 초진기록: 양식만 바꾸기 (P2)

두 번째 환자 P2(40세 여성)는 처음 왔습니다. 진료를 마친 의사가 남긴 짧은 메모입니다.

In [ ]:
memo = ka.load_note("P2")                # 진료 중 의사가 남긴 메모
print(memo)

같은 에이전트·같은 도구에 **양식**과 **의사 메모**만 바꿔 넣습니다.

이 메모는 문진표·기록과 **세 군데 어긋나도록** 만들어 두었습니다.
- 기저질환: 메모 "없음" ↔ 문진표 "빈혈"
- 음주: 메모 "회식 때 가끔" ↔ 문진표 "마시지 않음"
- 체중: 메모 "1년 4kg↑" ↔ 기록상 1년간 +0.9kg

초진기록 양식은 두 가지를 준비했습니다. 차이는 맨 끝의 **'메모·문진표 대조' 칸** 하나뿐입니다.

In [ ]:
print(ka.CHECK_BLOCK)                    # FORM_INITIAL = FORM_INITIAL_NOCHECK + 이 칸

In [ ]:
import re

def found_mismatch(out):
    """세 가지 불일치를 '확인 필요'로 표시했는지 줄 단위로 확인"""
    keys = {"기저질환(빈혈)": "빈혈|기저질환", "음주": "음주", "체중 변화": r"체중 변화|4\s*kg"}
    return {name: any(re.search(pat, line) and re.search("확인|불일치", line) for line in out.splitlines())
            for name, pat in keys.items()}

ctx = kt.Context("P2")                   # 두 번째 환자 P2를 연다 (오늘 할 일 = 초진기록)

# ① 대조 칸이 없는 양식
out_plain, _ = my_agent(ctx, ka.build_messages(ctx, memo, form=ka.FORM_INITIAL_NOCHECK))
# ② 대조 칸이 있는 양식 (form을 안 주면 ka.FORM_INITIAL)
out_checked, trace = my_agent(ctx, ka.build_messages(ctx, memo))

print("대조 칸 없음:", found_mismatch(out_plain))
print("대조 칸 있음:", found_mismatch(out_checked))

In [ ]:
print(out_checked)                       # 대조 칸이 있는 쪽의 전체 초안
ka.print_check(ka.check_output(ctx, out_checked, trace, memo))

"어긋나면 확인 필요라고 써라"라는 규칙은 시스템 프롬프트에 이미 있습니다. 그래도 대조 칸이 없으면 자주 놓칩니다(리허설: 평균 3곳 중 1곳).
**비교할 짝을 출력 칸으로 만들어 하나씩 쓰게 하면** 빠뜨리지 않습니다(리허설: 매번 3곳). LLM이 자주 놓치는 판단은 "잘 해라"가 아니라 **절차를 양식에 넣는 것**으로 안정시킵니다.

🔁 **선택**: 메모를 고쳐서(예: "술 안 마심"으로) 다시 돌려 보세요. 대조 결과가 어떻게 바뀌나요?

## 7. 치료 종료 기록 (P3)

세 번째 환자 P3는 위고비를 쓰다가 그만둔 환자입니다. 오늘이 마지막 진료입니다.
P3는 체중이 **5.7% 줄었지만** 위고비 1.0mg(유지 용량 1.7mg 전)에서 중단했습니다. 실습 정의상 약물 반응은 **'판정 불가'**입니다. 에이전트가 숫자에 끌려 '반응'이라고 쓰지 않는지 보세요.

In [ ]:
ctx = kt.Context("P3")                   # 세 번째 환자 P3을 연다 (오늘 할 일 = 치료 종료 기록)
out, trace = my_agent(ctx, ka.build_messages(ctx))
print(out)
ka.print_check(ka.check_output(ctx, out, trace))

판정은 LLM이 아니라 `assess_response` 도구가 정의대로 계산합니다. LLM의 역할은 그 결과를 **바꾸지 않고 옮기는 것**입니다.

In [ ]:
print(kt.assess_response(ctx))           # 에이전트가 옮겨 적은 판정의 원본

## 8. 채팅 화면으로 써 보기

같은 에이전트를 의사가 쓸 화면으로 띄웁니다. 환자를 고르고, P2는 메모를 고쳐 가며 실행해 보세요.

In [ ]:
app = ka.launch_app(client)       # 화면이 안 보이면 ka.launch_app(client, share=True) 로 공개 링크를 받으세요

In [ ]:
print(ka.usage_summary())         # 이 노트북에서 쓴 LLM 호출 수와 토큰

## 정리: 원내로 옮긴다면

| 오늘 실습 | 원내 배포 |
|---|---|
| Gemini 무료 API (합성 데이터) | 원내 GPU 서버의 오픈 웨이트 모델 (vLLM 등) — `base_url`·모델 이름 교체 |
| CSV 표 7개 | EMR 조회 API·데이터 웨어하우스. 도구 함수 안쪽만 교체 |
| `Context("P1")`: 연 환자에게만 묶인 도구 | 조회 권한·감사 로그와 함께 서버에서 보장 |
| 자동 점검 3종 | 배포 전 검증 + 운영 중 모니터링 지표 |

**토론**: 이 초안을 진료에 쓰려면 무엇이 더 필요할까요? (책임 소재, 의사 수정률 측정, 실패 사례 수집, 환자 동의 …)

⚠️ 무료 API에는 실제 환자 정보를 절대 넣지 마세요. 무료 등급의 입력은 서비스 개선에 쓰일 수 있습니다.